In [1]:
from transformers import (
    AutoTokenizer, TrainingArguments, Trainer, AutoModelForSeq2SeqLM, AutoConfig,
    T5Tokenizer, T5ForConditionalGeneration
)
from datasets import load_dataset, Dataset, DatasetDict
import torch
import pandas as pd
import numpy as np
from typing import Any, Union, List, Tuple, Dict
from transformers.data.data_collator import DataCollatorMixin, InputDataClass
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
import re

In [2]:
class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'

In [3]:
sel_cols = ['assetlongdescription_entity_llms', 'failurelocation_original', 
            'assetlongdescription_original', 'mode']
df = pd.read_csv('processed/asset2item.csv')[sel_cols]
df.rename({'assetlongdescription_original': 'answers.text'}, axis=1, inplace=True)
df['answers.text'] = pd.Series(df['answers.text'], dtype="string")

In [4]:
include_failure_locations = True

In [5]:
if include_failure_locations:
    for key in ['failurelocation_original', 'assetlongdescription_entity_llms']:
        df[key] = df[key].apply(lambda x: list(eval(x)))
    df['assetlongdescription_entity_llms'] = df['failurelocation_original'] + df['assetlongdescription_entity_llms']

In [6]:
df_train = df[df['mode']=='train']
df_val = df[df['mode']=='val']
df_test = df[df['mode']=='test']

In [7]:
# chkpt_path = '/dccstor/chrisconst2/AutoQA/fmea_recommender/entity_masking_mlm/checkpoint-14000'
chkpt_path = 'google/flan-t5-small'

In [8]:
tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-small')
config = AutoConfig.from_pretrained(chkpt_path, output_hidden_states=True)
model = AutoModelForSeq2SeqLM.from_pretrained(chkpt_path, config=config)

In [9]:
class CustomDataset(Dataset):
    def __init__(self, df):
        self.df = df
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        item = {
                'description': self.df.iloc[idx]['answers.text'],
                'failure_locations': self.df.iloc[idx]['failurelocation_original'],
                'labels': self.df.iloc[idx].assetlongdescription_entity_llms
               }
        return item

In [10]:
ds_train = CustomDataset(df_train)
ds_val = CustomDataset(df_val)
ds_test = CustomDataset(df_test)

In [11]:
def collate_fn(data, mask_prob=0.3):
    all_masked_texts = []
    all_labels = []
    batch = {'original_text': [], 'masked_text': [], 'failure_locations': []}
    for i in range(len(data)):
        failure_locations = data[i]['failure_locations'] 
        if include_failure_locations:
            text = f"{data[i]['description']}\nFailure locations:{failure_locations}"
        else:
            text = data[i]['description']
        failure_locations = 'Failure locations: ' + str(data[i]['failure_locations'])
        batch['original_text'].append(text)
        batch['failure_locations'].append(failure_locations)
        entities = np.array(data[i]['labels'])
        original_text = text
        sel_idxs = np.random.binomial(1, mask_prob, len(entities)).astype(bool)
        masked_entities = entities[sel_idxs]
        j = 0
        labels = ''
        masked_text = text
        for entity in masked_entities:
            while True:
                idx = masked_text.find(entity)
                if idx == -1:
                    break
                end_idx = idx + len(entity)
                masked_text = masked_text[:idx] + f"<extra_id_{j}> " + masked_text[end_idx:] + " "
                labels += f"<extra_id_{j}> {entity} "
                j += 1
        all_masked_texts.append(masked_text)
        all_labels.append(labels)
    masked_tokenized = tokenizer(all_masked_texts, padding="max_length", 
                                 max_length=512, truncation=True, return_tensors='pt')
    label_tokenized = tokenizer(all_labels, padding="max_length", 
                                max_length=512, truncation=True, return_tensors='pt')
    return {
        'input_ids': masked_tokenized['input_ids'],
        'labels': label_tokenized['input_ids']
    }

In [12]:
def compare_preds(item, outputs, limit=1):
    pred_str = tokenizer.batch_decode(torch.argmax(outputs, axis=2))
    masked_str = tokenizer.batch_decode(item['input_ids'])
    gt_str = tokenizer.batch_decode(item['labels'])
    for i in range(min(len(pred_str), limit)):
        print("Input"+"*"+"*"*100)
        print(masked_str[i].replace(tokenizer.pad_token, ''))
        print("*"*100)
        gt_str[i] = gt_str[i].replace(tokenizer.pad_token, '')
        pred_str[i] = pred_str[i].replace(tokenizer.pad_token, '')
        print(f'Predicted:{bcolors.OKGREEN}{pred_str[i]}{bcolors.ENDC}')
        print(f'    Label:{bcolors.OKCYAN}{gt_str[i]}{bcolors.ENDC}')

In [13]:
dataloader = DataLoader(ds_val, batch_size=4, collate_fn=collate_fn)

In [18]:
item = next(iter(dataloader))
outputs = model(**item)
compare_preds(item, outputs.logits, limit=2)

Input*****************************************************************************************************
The equipment Battery - Charger, is categorized as Electrical Asset and has the following boundary: The boundary of a typical battery charger for the purpose of this database is defined to include the following:,Battery Charger Input breakers are excluded, because PM for these can be found by referring to Motor Control Centers. Note, this program assumes that the battery charger is in nominally good condition to begin with. Battery chargers that have not been serviced for a long time may need to have a detailed inspection performed before this program is applied. Failure locations:['Transformer', 'Printed Circuit Boards (Firing circuit, Alarm, Auxiliary circuits, Current limit)', 'Fuse holder','<extra_id_0> ', 'Capacitors, Electrolytic', 'Timer','<extra_id_1> ', 'Filter Choke', 'Float and Equalize Potentiometers and Switches', 'Muffin fans', 'Input Fuse','<extra_id_2> ']</s>
*****

In [15]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [16]:
training_args = TrainingArguments(
    output_dir="entity_masking_mlm_flan",
    evaluation_strategy="epoch",
    num_train_epochs=10,
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=False,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    remove_unused_columns=False
    
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collate_fn,
)


Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
/u/chrisconst/.conda/envs/llm/lib/python3.10/site-packages/pydantic/_internal/_fields.py:151: UserWarning: Field "model_server_url" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/u/chrisconst/.conda/envs/llm/lib/python3.10/site-packages/pydantic/_internal/_config.py:322: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss



KeyboardInterrupt

